In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import numpy as np
import gseapy as gp

In [ ]:
homer_path = ''
env = ''

In [ ]:
os.system(env+'python ./metilene3/metilene3.py \
    -i ./data/GSE142241_cfMB.input.tsv \
    -o ./cfmb \
    -t 16 \
    -n 4 \
    -plot True \
')

In [ ]:
os.system('cp ./cfmb/DMRs-unsupervised.tsv ../SourceData/Fig.ED7a.txt')

In [ ]:
renameG = {
    'G0':'A',
    'G1':'B',
    'G2':'C',
    'G3':'D',
    'G4':'E',
    'G5':'F',
}

csf_s = pd.read_table('./cfmb/DMRs.tsv')
csf_s['hypomethylated'] = csf_s['Hypo-groups']
csf_s['intermediate'] = csf_s['Int-groups']
csf_s['hypermethylated'] = csf_s['Hyper-groups']
for i in renameG.keys():
    csf_s['hypomethylated'] = csf_s['hypomethylated'].str.replace(i,renameG[i])
    csf_s['intermediate'] = csf_s['intermediate'].str.replace(i,renameG[i])
    csf_s['hypermethylated'] = csf_s['hypermethylated'].str.replace(i,renameG[i])
csf_s['mode'] = 'supervised'

# csf = pd.concat([csf_u, csf_s]).sort_values(['mode','chr','start'])
csf = pd.concat([csf_s]).sort_values(['length','p-kwt'], ascending=[0,1])
csf = csf['chr	start	stop	meandiffabs	length	p-kwt	hypomethylated	intermediate	hypermethylated'.split('\t')]
csf.to_csv('./figures/ST6.tsv',sep='\t', index=False)
csf

In [ ]:
dmrmean_m_rename = pd.read_table('cfmb/heatmap.tsv', index_col=0)
colors = dmrmean_m_rename[[]]

colors['group'] = [i.split(' ')[0] for i in dmrmean_m_rename.index]
clsc = {
    'G0':sns.color_palette("Set2")[0],
    'G1':sns.color_palette("Set2")[1],
}
colors['subtype'] = dmrmean_m_rename.index.str.contains('NonTumor')
colors['groupc'] = colors['group'].map(clsc)
colors['subtypec'] = colors['subtype'].map({True:'green',False:'red','R132H':'orange'})
colors.head()

In [ ]:
from Bio import Phylo

tree = Phylo.read("./cfmb/DMTree.nwk", "newick")

def change_labels(clade):
    if clade.name:
        clade.name = clade.name.split('=')[-1]+'-'.join(['' for i in range(20)])
    for subclade in clade.clades:
        change_labels(subclade)

change_labels(tree.root)

cmap = colors['groupc'].to_dict()
for i in colors.index:
    cmap[i.split('=')[-1]+'-'.join(['' for i in range(20)])] = cmap[i]
f,a = plt.subplots(figsize=[6,2])
Phylo.draw(tree, axes=a, do_show=False, label_colors=cmap,show_confidence=True)
plt.xscale('symlog')
plt.xlim([-0.1,1e5/2])
a.spines['top'].set_visible(False)
a.spines['left'].set_visible(False)
a.spines['right'].set_visible(False)
a.yaxis.set_visible(False)
plt.savefig('./figures/ED7a.pdf', bbox_inches='tight')

In [ ]:
cm = sns.clustermap(dmrmean_m_rename,\
        row_colors=[colors['groupc'],\
                    colors['subtype'].map({True:'white',False:'white'}),\
                    colors['subtypec'],\
                    colors['subtype'].map({True:'white',False:'white'}),\
                    ],\
        # row_linkage=lk.linkage,\
        col_cluster=False,row_cluster=False,\
        cmap='Spectral_r', dendrogram_ratio=0.000001, xticklabels=False, yticklabels=False, \
        method='ward', cbar_pos=None, vmax=1, vmin=0, center=0.5, colors_ratio=0.03)

plt.savefig('./figures/ED7a-r.pdf', bbox_inches='tight')